# AI Data Analyst Agent — دموی جامع
این نوت‌بوک مسیر کامل پروژه را نشان می‌دهد: خواندن فایل فارسی/انگلیسی، پروفایل داده، تطبیق مفهومی ستون‌ها، تحلیل‌های آماری، نمودارهای متنوع و اجرای اختیاری ایجنت.

## 1. آماده‌سازی

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from app.agent.orchestrator import run_agent
from app.loaders.pandas_loader import load_data
from app.profiling.profiler import profile_dataset
from app.tools.analysis import analyze_data, resolve_column
from app.tools.visualization import create_chart

## 2. بارگذاری داده
لودر جداکننده و encoding فایل CSV را تشخیص می‌دهد و CSV، Excel و JSON را می‌خواند.

In [ ]:
samples = sorted((ROOT / 'uploads').glob('*'), key=lambda path: path.stat().st_size, reverse=True)
if not samples:
    raise FileNotFoundError('یک فایل CSV یا Excel در پوشه uploads قرار دهید.')

data_path = samples[0]
df = load_data(data_path)
print(data_path.name, df.shape)
df.head()

## 3. پروفایل سریع و تطبیق هوشمند ستون

In [ ]:
profile_dataset(df)

In [ ]:
# «سن» می‌تواند به ستون Age و «فشار خون» به BloodPressure نگاشت شود.
for concept in ('سن', 'فشار خون', 'bmi'):
    try:
        print(f'{concept} → {resolve_column(df, concept)}')
    except ValueError as error:
        print(error)

## 4. تحلیل‌های آماری
روش‌های موجود: `describe`، `missing`، `correlation`، `outliers`، `frequency`، `group` و `trend`.

In [ ]:
numeric = df.select_dtypes('number').columns.tolist()
for method in ('describe', 'missing', 'outliers'):
    result = analyze_data(df, method, numeric[:3] or None)
    print(f'\n--- {method} ---')
    print(result)

if len(numeric) >= 2:
    analyze_data(df, 'correlation', numeric[:4])

## 5. گالری نمودارها
انواع موجود: histogram، box، scatter، line، bar، pie و correlation. برچسب‌های فارسی پیش از ذخیره برای Matplotlib آماده می‌شوند.

In [ ]:
from IPython.display import Image, display

charts = []
if numeric:
    charts += [create_chart(df, 'histogram', x=numeric[0]), create_chart(df, 'box', x=numeric[0])]
if len(numeric) >= 2:
    charts += [create_chart(df, 'scatter', x=numeric[0], y=numeric[1]), create_chart(df, 'correlation')]

for chart in charts:
    print(chart['title'])
    display(Image(chart['path']))

## 6. اجرای اختیاری ایجنت چندمرحله‌ای
پس از تنظیم `OPENROUTER_API_KEY` مقدار زیر را `True` کنید. ایجنت همهٔ بخش‌های درخواست را اجرا و پاسخ Markdown را هم‌زبان کاربر تولید می‌کند.

In [ ]:
RUN_LIVE_AGENT = False

if RUN_LIVE_AGENT:
    response = run_agent(df, 'سن را توصیف کن، نقاط پرت را بررسی کن و هیستوگرام و نمودار جعبه‌ای بساز.')
    print(response.answer)
    print([chart.path for chart in response.charts])
else:
    print('برای جلوگیری از مصرف API، اجرای زنده غیرفعال است.')

## 7. دموی وب
در ریشهٔ پروژه اجرا کنید: `streamlit run streamlit_app.py`. همین فایل نقطهٔ ورود Streamlit Community Cloud است و به backend جداگانه نیاز ندارد.